# NB01 — subclass closure
Start with one assertion: Alice works for Acme. A domain axiom infers Employee membership;
a subclass axiom then infers Person membership. The closure should have exactly three facts.
This lab needs only this GestaltDB checkout, not an embedded backend or RDF parser.

In [ ]:
from gestaltdb.ontology import Domain, Ontology, SubClassOf, TYPE_PREDICATE, compile_ontology
from gestaltdb.reasoning import DatalogEngine, Fact, ReasoningLimits

plan = compile_ontology(Ontology('people', [
    Domain('worksFor', 'Employee'),
    SubClassOf('Employee', 'Person'),
]))
asserted = Fact('alice', 'worksFor', 'acme')
result = DatalogEngine().evaluate(plan.program, [asserted], limits=ReasoningLimits(), collect_proofs=True)
person = Fact('alice', TYPE_PREDICATE, 'Person')
assert set(result.facts) == {asserted, Fact('alice', TYPE_PREDICATE, 'Employee'), person}
assert Fact('alice', TYPE_PREDICATE, 'Researcher') not in result
assert not result.explain(person).truncated
for fact in result.facts:
    print(fact)
print(result.explain(person))


The original assertion remains in the result. No labels were written to any database.
Exercise: remove the Domain axiom. Why does SubClassOf alone no longer infer Person?
Answer: nothing supplies the Employee premise. Employment alone does not establish a research role.

## A research role requires another premise
Researcher is a refinement of Employee. Join Employee membership with an explicit job-description category.
The categories researcher and not_researcher are named entities in this binary example, not datatype literals.
No description does not imply not_researcher. A description alone does not establish employment.

In [ ]:
from gestaltdb.reasoning import Atom, Program, Rule, Variable
refined = compile_ontology(Ontology('jobs', [*plan.ontology.axioms, SubClassOf('Researcher', 'Employee')]))
x = Variable('x')
rule = Rule('employee-with-research-job', [
    Atom(x, TYPE_PREDICATE, 'Employee'), Atom(x, 'hasJobDescription', 'researcher'),
], Atom(x, TYPE_PREDICATE, 'Researcher'))
program = Program([*refined.program.rules, rule])
researcher = Fact('alice', TYPE_PREDICATE, 'Researcher')
engine = DatalogEngine()
for description in (None, 'not_researcher', 'researcher'):
    inputs = [asserted]
    if description is not None:
        inputs.append(Fact('alice', 'hasJobDescription', description))
    classified = engine.evaluate(program, inputs, limits=ReasoningLimits())
    assert Fact('alice', TYPE_PREDICATE, 'Employee') in classified
    assert (researcher in classified) == (description == 'researcher')
job_only = engine.evaluate(program, [Fact('alice', 'hasJobDescription', 'researcher')], limits=ReasoningLimits())
assert researcher not in job_only
assert Fact('alice', TYPE_PREDICATE, 'Employee') not in job_only
